In [ ]:
import numpy as np
import random as rd
import plotly.graph_objects as go
from scipy.stats import poisson

#### Example 4.2 : Jack's Car Rental

In [ ]:
# Parameters
maxCars = 20
maxMoves = 5
nStatesJCR = (maxCars + 1) ** 2
freeRide = False
maxParkingSpace = 20
parkingFee = 4
movingFee = 2
lambdaRentLoc1, lambdaRentLoc2, lambdaRetLoc1, lambdaRetLoc2 = 3, 4, 3, 2

We start by defining the probabilities $P^{inter}_1, P^{inter}_2$ to go from one intermediary state $s_{inter}$ to a final state $s'$ at locations 1 and 2 respectively. Intermediary states incode the number of cars present at locations 1 and 2 after all overnight movements are completed. Final states incode the number of cars present at locations 1 and 2 at the end of the work day, so that $P^{inter}_1, P^{inter}_2$ are the probabilities to start a day at $s^{inter}$ and end it at $s'$.

Let $n^{inter}_1, n^{inter}_2, n'_1, n'_2$ be # of cars at each location of each state. Since the two locations are independent, $p^{inter}_1$ and $p^{inter}_2$ can be computed independently.

For a location $i \in \{1, 2\}$, we define the probability to start the day with $n^{inter}$ and end it with $n'$ cars is :
$$P^{inter}(n^{inter}, n') = 
    \begin{cases}
        \displaystyle\sum_{k=(n^{inter} - n')^+}^{n^{inter}} p(k, \lambda^{rent}) p(k + n' - n^{inter}, \lambda^{return}) + \underbrace{\sum_{k \geq n^{inter} + 1} p(k, \lambda^{rent})}_{\substack{\text{probability of rent demand} \\ \text{exceeding the available number of cars } n^{inter}}} p(n', \lambda^{return}) & \text{if } n' < n_{max} \\
        \displaystyle\sum_{k=(n^{inter} - n')^+}^{n^{inter}} p(k, \lambda^{rent}) \underbrace{\sum_{l \geq k + n' - n^{inter}} p(l, \lambda^{return})}_{\substack{\text{probability of retuned cars} \\ \text{exceeding final available space (after rent)}}} + \sum_{k \geq n^{inter} + 1} p(k, \lambda^{rent}) \sum_{l \geq n'} p(l, \lambda^{return}) & \text{if } n' = n_{max}
    \end{cases} $$
where $p(k, \lambda) = \frac{\lambda^k}{k!} e^{-\lambda}$ is the poisson mass function of demand of rent or number of returns at a given location at rate $\lambda$.


Computing $p^{inter}_1$ and $p^{inter}_2$ we can deduce the probability of going from an intermediary state $s^{inter} := (n^{inter}_1, n'_1)$ to a final state $s' := (n'_1, n'_2)$ as $$P^{inter}(s^{inter}, s') = P^{inter}_1(n^{inter}_1, n'_1) \times P^{inter}_2(n^{inter}_2, n'_2)$$

Similarly, we compute rent revenue according to the expected number of rents at each locations starting from $s^{inter}$ and arriving at any possible state $n'$. Since any number of car rent demand has a positive probability, and since it is independent from number of returned cars, expected rent revenue is simply computed as:
$$ r^{rent}(n^{inter}) = c^{rent} \left ( \displaystyle\sum_{k=0}^{n^{inter}}kp(k, \lambda^{rent}) + n^{inter} \displaystyle\sum_{k\geq n^{inter}+1} p(k, \lambda^{rent}) \right )$$
So that the expected rent revenue over all locations is simply $R^{rent}(s^{inter}) = r^{rent}_1(n^{inter}_1) + r^{rent}_2(n^{inter}_2)$

Next step is to define legal moves for each state $s$ in order to compute the total expected reward function. Let $s := (n_1, n_2)$. Let $a$ be the number of cars moved from location 1 to location 2. Taking into account $n_{max}$ (the maximum number of cars that can be parked at a location) and $a_{max}$ (the maximum number of cars that can be moved from one location to the other), we have that :
$$ a \in \llbracket -\min(n_2, a_{max}, n_{max} - n_1), \min(n_1, a_{max}, n_{max} - n_2) \rrbracket$$

So that the total expected reward function can be computed as :
$$R(s, a) = R^{rent}(n_1-a, n_2+a) - c^{moves} |a|$$
In the modified versions of the problem, one might want to take into account the free ride that the employee proposes, or the cost of using a second parking lot when limitied space if available. These kind of nonlinearities are easily incorporated to the expected reward function with :
$$r^{free \ ride}(s, a) = 
    \begin{cases}
        c^{moves} & \text{if } a \geq 1 \\
        0 & \text{otherwise}
    \end{cases}$$

$$r^{parking}_i(n_i, a) = 
    \begin{cases}
        -c^{parking} & \text{if } n_i \pm a > n^{parking}_{max} \\
        0 &  \text{otherwise}
    \end{cases}
$$

Finally, the dynamics $P$ of the MDP can be computed thanks to $P^{inter}$ where :
$$P(s, a, s') = \displaystyle\sum_{s' \in S} P^{inter}(s + a, s')$$
where $s + a = (n_1 - a, n_2 + a)$

In [ ]:
# Build JCR problem
def buildJCRModel(maxCars, maxMoves, movingFee, lambdaRentLoc1, lambdaRentLoc2, lambdaRetLoc1, lambdaRetLoc2, freeRide, maxParkingSpace, parkingFee):
    nStates = (maxCars + 1) ** 2
    actionMapping = [np.arange(-min(min(nCarsLoc2, maxMoves), maxCars - nCarsLoc1) + maxMoves, min(min(nCarsLoc1, maxMoves), maxCars - nCarsLoc2) + 1 + maxMoves) for nCarsLoc1 in range(maxCars + 1) for nCarsLoc2 in range(maxCars + 1)]

    # PInterLoc1, PInterLoc2 : probability of going from nCarsInter to nCarsFinal after all moving of cars at Loc1 and Loc2 respectively
    PInterLoc1 = np.zeros((maxCars + 1, maxCars + 1))
    PInterLoc2 = np.zeros((maxCars + 1, maxCars + 1))
    for nCarsInter in range(maxCars + 1):
        for nCarsFinal in range(maxCars + 1):
            delta = nCarsFinal - nCarsInter
            if nCarsFinal < maxCars:
                for nCarsRented in range(max(0, -delta), nCarsInter + 1):
                    PInterLoc1[nCarsInter, nCarsFinal] += poisson.pmf(nCarsRented, lambdaRentLoc1) * poisson.pmf(nCarsRented + delta, lambdaRetLoc1)
                    PInterLoc2[nCarsInter, nCarsFinal] += poisson.pmf(nCarsRented, lambdaRentLoc2) * poisson.pmf(nCarsRented + delta, lambdaRetLoc2)
                PInterLoc1[nCarsInter, nCarsFinal] += poisson.pmf(nCarsFinal, lambdaRetLoc1) * (1 - poisson.cdf(nCarsInter, lambdaRentLoc1))
                PInterLoc2[nCarsInter, nCarsFinal] += poisson.pmf(nCarsFinal, lambdaRetLoc2) * (1 - poisson.cdf(nCarsInter, lambdaRentLoc2))

            else:
                for nCarsRented in range(nCarsInter + 1):
                    PInterLoc1[nCarsInter, maxCars] += poisson.pmf(nCarsRented, lambdaRentLoc1) * (1 - poisson.cdf(nCarsRented + delta - 1, lambdaRetLoc1))
                    PInterLoc2[nCarsInter, maxCars] += poisson.pmf(nCarsRented, lambdaRentLoc2) * (1 - poisson.cdf(nCarsRented + delta - 1, lambdaRetLoc2))
                PInterLoc1[nCarsInter, maxCars] += (1 - poisson.cdf(nCarsInter, lambdaRentLoc1)) * (1 - poisson.cdf(nCarsFinal - 1, lambdaRetLoc1))
                PInterLoc2[nCarsInter, maxCars] += (1 - poisson.cdf(nCarsInter, lambdaRentLoc2)) * (1 - poisson.cdf(nCarsFinal - 1, lambdaRetLoc2))

    # PInter
    PInter = np.zeros((nStates, nStates))
    for s1 in range(nStates):     
        for s2 in range(nStates):         
            nCarsInterLoc1, nCarsInterLoc2 = s1 // (maxCars + 1), s1 % (maxCars + 1)         
            nCarsFinalLoc1, nCarsFinalLoc2 = s2 // (maxCars + 1), s2 % (maxCars + 1)         
            PInter[s1, s2] = PInterLoc1[nCarsInterLoc1, nCarsFinalLoc1] * PInterLoc2[nCarsInterLoc2, nCarsFinalLoc2]

    # Transition : transition from state s to state sInter after moving a cars from Loc1 to Loc2
    nActionMax = 2 * maxMoves + 1
    transition = -np.ones((nStates, nActionMax), dtype=int)
    for s in range(nStates):     
        nCarsInterLoc1, nCarsInterLoc2 = s // (maxCars + 1), s % (maxCars + 1)     
        for a in actionMapping[s]:         
            transition[s, a] = nCarsInterLoc2 + a - maxMoves + (maxCars + 1) * (nCarsInterLoc1 - a + maxMoves)

    # RInter : expected rental revenue when starting the day at s (intermediary state after all moving of cars)
    RInter = np.zeros(nStates)
    for s in range(nStates):
        nCarsLoc1, nCarsLoc2 = s // (maxCars + 1), s % (maxCars + 1)
        rewardLoc1 = sum([10 * nCarsRented * poisson.pmf(nCarsRented, lambdaRentLoc1) for nCarsRented in range(nCarsLoc1 + 1)]) + 10 * nCarsLoc1 * (1 - poisson.cdf(nCarsLoc1, lambdaRentLoc1))
        rewardLoc2 = sum([10 * nCarsRented * poisson.pmf(nCarsRented, lambdaRentLoc2) for nCarsRented in range(nCarsLoc2 + 1)]) + 10 * nCarsLoc2 * (1 - poisson.cdf(nCarsLoc2, lambdaRentLoc2))
        RInter[s] = rewardLoc1 + rewardLoc2

    # Rewards : moving cost (and extra parking costs) + expected rental revenue
    rewardMapping = np.nan * np.ones((nStates, nActionMax))
    for s in range(nStates):
        for a in actionMapping[s]:
            sInter = transition[s, a]
            nCarsInterLoc1, nCarsInterLoc2 = sInter // (maxCars + 1), sInter % (maxCars + 1)
            rewardMapping[s, a] = -movingFee * abs(a - maxMoves) + freeRide * movingFee * (a - maxMoves >= 1) + RInter[sInter] + parkingFee * ((nCarsInterLoc1 > maxParkingSpace) + (nCarsInterLoc2 > maxParkingSpace))

    # Dynamics : transition probabilities of the MDP
    dynamics = np.zeros((nStates, nActionMax, nStates))
    for s in range(nStates):
        for a in actionMapping[s]:
            for sPrime in range(nStates):
                dynamics[s, a, sPrime] = PInter[transition[s, a], sPrime]

    # Terminal states
    terminals = []

    # Legal action mask
    legalActions = np.zeros((nStates, nActionMax), dtype=bool)
    for state, action in enumerate(actionMapping):
        legalActions[state, action] = True

    return nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals


In [ ]:
def plotPolicyJCR(policy):
    x = np.arange(maxCars + 1)
    y = np.arange(maxCars + 1)

    X, Y = np.meshgrid(x, y)
    Z = policy[X + (maxCars + 1) * Y] - maxMoves

    fig = go.Figure()
    fig.add_trace(go.Contour(x=x, y=x, z=Z, contours=dict(coloring="none", showlabels=True, start=-maxMoves, end=maxMoves, size=1), line=dict(color="black")))
    fig.update_layout(width=400, height=400)
    fig.show()

In [ ]:
def plotValueJCR(V):
    x = np.arange(maxCars + 1)
    y = np.arange(maxCars + 1)

    X, Y = np.meshgrid(x, y)
    Z = V[X + (maxCars + 1) * Y]

    fig = go.Figure()
    fig.add_trace(go.Surface(x=x, y=y, z=Z))
    fig.update_layout(width=400, height=400)
    
    fig.show()

In [ ]:
# Model
modelJCR = buildJCRModel(maxCars, maxMoves, movingFee, lambdaRentLoc1, lambdaRentLoc2, lambdaRetLoc1, lambdaRetLoc2, freeRide, maxParkingSpace, parkingFee)
gammaJCR = 0.9

In [ ]:
# Policy Evaluation
def Ld(model, s, a, gamma, V):
    nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals = model
    return rewardMapping[s, a] + gamma * sum(dynamics[s, a, sPrime] * V[sPrime] for sPrime in np.where(np.sum(legalActions, axis=1) >= 1)[0])


def policyEval(model, policy, V, theta, gamma):
    nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals = model
    converged = False
    while not converged:
        delta = 0
        for s in np.where(np.sum(legalActions, axis=1) >= 1)[0]:
            v = V[s]
            V[s] = Ld(model, s, policy[s], gamma, V)
            delta = max(delta, abs(v - V[s]))
        converged = delta < theta


# Policy Improvement
def policyImprov(model, policy, V, gamma):
    policyStable = True
    nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals = model
    for s in np.where(np.sum(legalActions, axis=1) >= 1)[0]:
        oldPolicy = policy[s]
        # Greedy action improv
        maxi = Ld(model, s, oldPolicy, gamma, V)
        maxiAction = policy[s]
        for a in actionMapping[s]:
            qValAction = Ld(model, s, a, gamma, V)
            if qValAction > maxi:
                maxi = qValAction
                maxiAction = a

        policy[s] = maxiAction
        
        if policy[s] != oldPolicy:
            policyStable = False
    return policyStable

def policyIter(model, theta, gamma, policyInit, vInit):
    # Initialization
    nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals = model
    V = vInit.copy()
    V[terminals] = np.zeros(len(terminals))
    policy = policyInit.copy()
    policyStable = False
    iter = 0
    while not policyStable:
        # Policy Evaluation
        print(f"iteration {iter}")
        policyEval(model, policy, V, theta, gamma)

        # Policy Improv
        policyStable = policyImprov(model, policy, V, gamma)

        iter += 1

    return V, policy


In [ ]:
epsilonJCR = 0.001
policyInitJCR = maxMoves * np.ones(nStatesJCR, dtype=int)
vInitJCR = np.zeros(nStatesJCR)
vStarJCR, policyStarJCR = policyIter(modelJCR, epsilonJCR, gammaJCR, policyInitJCR, vInitJCR)
plotPolicyJCR(policyStarJCR)
plotValueJCR(vStarJCR)

#### Example 4.3 : Gambler's Problem

In [ ]:
def plotPolicyGP(policies):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=[rd.choice(actions) for actions in policies]))    
    fig.show()

In [ ]:
def plotValueGP(V):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=V[:-2]))    
    fig.show()

In [ ]:
def buildGPModel(goal, ph):
    nStates = goal + 1
    actionMapping = [np.arange(min(s, goal - s) + 1) for s in range(nStates)]
    nActionMax = goal + 1
    
    dynamics = np.zeros((nStates, nActionMax, nStates))
    for s in range(nStates):
        if s == 0 or s == goal:
            dynamics[s, :, s] = np.ones(nActionMax)
        if 0 < s < goal:
            for a in actionMapping[s]:
                dynamics[s, a, s + a] = ph
                dynamics[s, a, s - a] = 1 - ph

    rewardMapping = np.zeros((nStates, nActionMax))
    for s in range(nStates):
        if s < goal:
            for a in actionMapping[s]:
                rewardMapping[s, a] = dynamics[s, a, goal]

    terminals = [0, goal]

    legalActions = np.zeros((nStates, nActionMax), dtype=bool)
    for state, action in enumerate(actionMapping):
        legalActions[state, action] = True

    return nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals

In [ ]:
def valueIter(model, theta, gamma, vInit):
    # Initialization
    nStates, legalActions, actionMapping, rewardMapping, dynamics, terminals = model
    V = vInit.copy()
    V[terminals] = np.zeros(len(terminals))
    iter = 0
    converged = False
    while not converged:
        print(f"iteration {iter}")
        policy = []
        delta = 0
        for s in np.where(np.sum(legalActions, axis=1) >= 1)[0]:
            v = V[s]
            # Greedy action improv
            maxi = None
            for a in actionMapping[s]:
                qValAction = Ld(model, s, a, gamma, V)
                if maxi is None or qValAction > maxi:
                    maxi = qValAction
                    argmax = [a]
                elif qValAction == maxi:
                    argmax.append(a)

            policy.append(argmax)
            V[s] = qValAction
            delta = max(delta, abs(v - V[s]))

        converged = delta < theta      

        iter += 1

    return V, policy

In [ ]:
goal = 100
ph = 0.99
gammaGP = 1
modelGP = buildGPModel(goal, ph)
vInitGP = np.zeros(goal + 1)
epsilonGP = 0.001
vStarGP, policyStarGP = valueIter(modelGP, epsilonGP, gammaGP, vInitGP)

In [ ]:
plotPolicyGP(policyStarGP)
plotValueGP(vStarGP)